# How to build an AI Agent with PydanticAI and the ClickHouse MCP Server

In this notebook we'll see how to build an [PydanticAI](https://ai.pydantic.dev/mcp/client/#__tabbed_1_1) AI agent that can interact with [ClickHouse's SQL playground](https://sql.clickhouse.com/) using [ClickHouse's MCP Server](https://github.com/ClickHouse/mcp-clickhouse).


Updated dependencies: 5 September 2026. Outputs are cleared; see [setup and validation](../README.md).

## Install libraries
We need to install the PydanticAI library.

In [ ]:
%pip install -r requirements.txt


In [ ]:
# Dependencies installed above.


## Setup credentials
Let's provide our Anthropic API key.

In [ ]:
import os, getpass

In [ ]:
key_name = "OPENAI_API_KEY" if os.getenv("LLM_PROVIDER", "anthropic") == "openai" else "ANTHROPIC_API_KEY"
if not os.getenv(key_name):
    os.environ[key_name] = getpass.getpass(f"Enter {key_name}: ")


We'll also define the credentials to connect to the ClickHouse SQL playground:

In [ ]:
import os
env = {
    "CLICKHOUSE_HOST": os.getenv("CLICKHOUSE_HOST", 'sql-clickhouse.clickhouse.com'),
    "CLICKHOUSE_PORT": os.getenv("CLICKHOUSE_PORT", '8443'),
    "CLICKHOUSE_USER": os.getenv("CLICKHOUSE_USER", 'demo'),
    "CLICKHOUSE_PASSWORD": os.getenv("CLICKHOUSE_PASSWORD", ''),
    "CLICKHOUSE_SECURE": os.getenv("CLICKHOUSE_SECURE", 'true')
}

## Initialize MCP Server and PydanticAI agent

Lets configure the ClickHouse MCP Server to point at the ClickHouse SQL playground and also initialize our PydanticAI agent and ask it a question:

In [ ]:
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset
from fastmcp import Client
from fastmcp.client.transports import StdioTransport


In [ ]:
server = MCPToolset(Client(StdioTransport(
    "uv", ["tool", "run", "--python", "3.13", "--from", "mcp-clickhouse==0.6.0", "mcp-clickhouse"],
    env=env,
), timeout=60))
model = ("openai:" + os.getenv("OPENAI_MODEL", "gpt-5.6-luna")
         if os.getenv("LLM_PROVIDER", "anthropic") == "openai"
         else "anthropic:" + os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5"))
agent = Agent(model, toolsets=[server])


## Ask the agent a question

And now let's ask the agent a question:

In [ ]:
async with agent:
    result = await agent.run(os.getenv("MCP_PROMPT", "Who's done the most PRs for ClickHouse?"))
    print(result.output)
